In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):

        def _GetPrototype(self, descriptor):
            return self.GetMessageClass(descriptor)

        message_factory.MessageFactory.GetPrototype = _GetPrototype
except Exception:
    pass

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split  # added for explicit split

base_dir = "/kaggle/input/petfinder-pawpularity-score"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

train_csv = pd.read_csv(os.path.join(base_dir, "train.csv"))
train_csv = train_csv.rename(columns=lambda x: x.strip())
test_csv = pd.read_csv(os.path.join(base_dir, "test.csv"))
test_csv = test_csv.rename(columns=lambda x: x.strip())
submission = pd.read_csv(os.path.join(base_dir, "sample_submission.csv"))
submission = submission.rename(columns=lambda x: x.strip())

train_feat = train_csv.drop(columns=["Id", "Pawpularity"])
train_target = train_csv["Pawpularity"]




2026-05-05 00:42:14.910552: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777941734.923798      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777941734.928188      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 00:42:14.946583: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
train_imgs = []
train_ids = []
for img_id in train_csv["Id"]:
    img_path = os.path.join(train_dir, f"{img_id}.jpg")
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
    train_imgs.append(img.astype(np.float32) / 255.0)
    train_ids.append(img_id)

train_imgs = np.stack(train_imgs)

mask_train = train_csv["Id"].isin(train_ids)
train_feat = train_feat[mask_train].reset_index(drop=True)
train_target = train_target[mask_train].reset_index(drop=True)

id_to_index = {img_id: idx for idx, img_id in enumerate(train_ids)}
img_indices = [
    id_to_index[img_id]
    for img_id in train_feat.index.map(lambda i: train_csv.loc[i, "Id"])
]
X_img_filtered = train_imgs[img_indices]

X_csv_train, X_csv_val, X_img_train, X_img_val, y_train, y_val = train_test_split(
    train_feat.values,
    X_img_filtered,
    train_target.values,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)




In [3]:
test_ids = list(test_csv["Id"])
num_test = len(test_ids)
test_imgs = np.zeros((num_test, 64, 64, 3), dtype=np.float32)

for idx, img_id in enumerate(test_ids):
    img_path = os.path.join(test_dir, f"{img_id}.jpg")
    img = cv2.imread(img_path)
    if img is None:
        continue  # keep zero placeholder
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
    test_imgs[idx] = img.astype(np.float32) / 255.0




In [4]:
norm_layer = tf.keras.layers.Normalization()
norm_layer.adapt(train_feat.values)

csv_input = tf.keras.Input(shape=(train_feat.shape[1],), name="CSV_Input")
x = norm_layer(csv_input)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
x = tf.keras.layers.Dense(200, activation="elu", kernel_initializer="he_normal")(x)
csv_branch = tf.keras.layers.Dropout(0.5)(x)

img_input = tf.keras.Input(shape=(64, 64, 3), name="IMG_Input")
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(img_input)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.MaxPooling2D(4)(y)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.MaxPooling2D(4)(y)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.MaxPooling2D(3)(y)
y = tf.keras.layers.Dropout(0.5)(y)
y = tf.keras.layers.Conv2D(
    120, 4, padding="same", activation="elu", kernel_initializer="he_normal"
)(y)
y = tf.keras.layers.Flatten()(y)
y = tf.keras.layers.Dense(
    300, activation="elu", kernel_initializer="he_normal", use_bias=False
)(y)
y = tf.keras.layers.Dropout(0.5)(y)
y = tf.keras.layers.Dense(
    300, activation="elu", kernel_initializer="he_normal", use_bias=False
)(y)
y = tf.keras.layers.Dropout(0.5)(y)
img_branch = y

csv_output = tf.keras.layers.Dense(1, name="CSV_Output")(csv_branch)
img_output = tf.keras.layers.Dense(1, name="IMG_Output")(img_branch)

model = tf.keras.Model(
    inputs=[csv_input, img_input],
    outputs=[csv_output, img_output],
    name="Pawpularity_Model",
)




2026-05-05 00:43:07.385542: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777941787.386004      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


In [5]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.002, decay_steps=10000, decay_rate=0.97
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
model.compile(
    loss=["mse", "mse"],
    loss_weights=[0.5, 0.5],
    optimizer=optimizer,
    metrics=[
        tf.keras.metrics.RootMeanSquaredError(),
        tf.keras.metrics.RootMeanSquaredError(),
    ],
)




In [6]:
model.fit(
    x=[X_csv_train, X_img_train],
    y=[y_train, y_train],
    epochs=30,  # increased epochs for modest performance gain
    batch_size=64,
    validation_data=([X_csv_val, X_img_val], [y_val, y_val]),
    verbose=2,
)




Epoch 1/30


I0000 00:00:1777941792.824251     110 service.cc:148] XLA service 0x7feb68028b80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777941792.824286     110 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 00:43:12.882345: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777941793.154681     110 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1777941797.781692     110 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


112/112 - 17s - 148ms/step - CSV_Output_loss: 499.9408 - CSV_Output_root_mean_squared_error: 22.3732 - IMG_Output_loss: 506.0140 - IMG_Output_root_mean_squared_error: 22.5034 - loss: 503.4814 - val_CSV_Output_loss: 466.4977 - val_CSV_Output_root_mean_squared_error: 21.5883 - val_IMG_Output_loss: 459.8125 - val_IMG_Output_root_mean_squared_error: 21.4318 - val_loss: 462.6894
Epoch 2/30
112/112 - 2s - 21ms/step - CSV_Output_loss: 445.7303 - CSV_Output_root_mean_squared_error: 21.1309 - IMG_Output_loss: 446.4202 - IMG_Output_root_mean_squared_error: 21.1449 - loss: 446.8103 - val_CSV_Output_loss: 459.6883 - val_CSV_Output_root_mean_squared_error: 21.4283 - val_IMG_Output_loss: 460.4934 - val_IMG_Output_root_mean_squared_error: 21.4488 - val_loss: 459.6128
Epoch 3/30
112/112 - 2s - 21ms/step - CSV_Output_loss: 437.5976 - CSV_Output_root_mean_squared_error: 20.9325 - IMG_Output_loss: 444.1668 - IMG_Output_root_mean_squared_error: 21.0998 - loss: 441.6842 - val_CSV_Output_loss: 459.5602 - va

In [7]:
# Prepare test metadata (CSV features) for prediction
test_feat = test_csv.drop(columns=["Id"])

# Evaluate optimal weighting on validation predictions
val_csv_pred, val_img_pred = model.predict([X_csv_val, X_img_val], verbose=0)
val_csv_pred = val_csv_pred.squeeze()
val_img_pred = val_img_pred.squeeze()
denom = np.sum((val_csv_pred - val_img_pred) ** 2)
if denom > 0:
    w_opt = np.sum((y_val - val_img_pred) * (val_csv_pred - val_img_pred)) / denom
else:
    w_opt = 0.5
w_opt = np.clip(w_opt, 0.0, 1.0)  # keep weight within [0,1]

# Predict on test data
csv_pred, img_pred = model.predict([test_feat.values, test_imgs], verbose=0)
final_pred = w_opt * csv_pred.squeeze() + (1 - w_opt) * img_pred.squeeze()
final_pred = np.clip(final_pred, 0.0, 100.0)

final_df = pd.DataFrame({"Id": test_ids, "Pawpularity": final_pred})

os.makedirs("/kaggle/working", exist_ok=True)
output_path = "/kaggle/working/submission.csv"
final_df.to_csv(output_path, index=False)
print(f"Submission written to {output_path}")

Submission written to /kaggle/working/submission.csv
